# 🐍 Управление файловой системой: `pathlib` vs `os`

🎯 **Цели:** 

- Научиться безопасно создавать, читать, искать и перемещать файлы
- Понять разницу между строковыми путями и объектами `Path`
- Избежать типичных багов кроссплатформенности и `PermissionError`

In [ ]:
import os
import shutil
from pathlib import Path
import json

# 🔧 Инициализация безопасной рабочей директории
WORKSPACE = Path("demo_workspace")
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)

# Создаём тестовую структуру для демонстраций
(WORKSPACE / "raw_data").mkdir()
(WORKSPACE / "raw_data" / "log_01.csv").write_text("id,val\n1,10\n2,20")
(WORKSPACE / "raw_data" / "log_02.json").write_text('{"status": "ok"}')
(WORKSPACE / "raw_data" / "img.png").write_bytes(b"\x89PNG\r\n\x1a\n")
(WORKSPACE / "archive").mkdir()

print(f"✅ Рабочая папка готова: {WORKSPACE.resolve()}")

## 🔹 1. Путь как объект, а не строка
Раньше: `os.path.join("a", "b")` или `"a/" + "b"`  
Сейчас: `Path("a") / "b"` → кроссплатформенно, типизировано, с методами.

In [ ]:
# Базовое создание пути
p = Path("demo_workspace", "raw_data", "log_01.csv")
print(f"Объект: {p}")
print(f"Абсолютный: {p.resolve()}")
print(f"Имя файла: {p.name}")
print(f"Без расширения: {p.stem}")
print(f"Расширение: {p.suffix}")
print(f"Родительская папка: {p.parent}")

# Магический оператор /
base = Path("demo_workspace")
cfg = base / "config" / "settings.yaml"
print(f"\nСборка через '/': {cfg}")
# ✅ Автоматически использует \ на Windows, / на Linux/macOS

## 🔹 2. Интроспекция и создание ресурсов
Не угадывайте состояние файловой системы. Спрашивайте у `Path`.

In [ ]:
p_dir = Path("demo_workspace/archive")
p_file = Path("demo_workspace/raw_data/log_01.csv")
p_fake = Path("demo_workspace/missing.txt")

print("📂 archive/ является директорией?", p_dir.is_dir())
print("📄 log_01.csv является файлом?", p_file.is_file())
print("🕳️ missing.txt существует?", p_fake.exists())

# Создание вложенной структуры одной командой
new_path = Path("demo_workspace") / "reports" / "2026" / "Q1"
new_path.mkdir(parents=True, exist_ok=True)
print(f"\n✅ Папка создана: {new_path.exists()}")
# parents=True → создаёт 2026 и Q1 автоматически
# exist_ok=True → не кидает FileExistsError, если уже есть

## 🔹 3. Чтение/Запись без `with open()`
Для небольших файлов и конфигов `pathlib` экономит код.

In [ ]:
# Запись текста (перезаписывает файл)
note = Path("demo_workspace/notes.txt")
note.write_text("Строка 1\nСтрока 2\n", encoding="utf-8")

# Чтение всего содержимого
print("📖 Чтение:")
print(note.read_text(encoding="utf-8"))

# Добавление в конец (для больших файлов/логов используйте .open())
with note.open("a", encoding="utf-8") as f:
    f.write("Строка 3\n")
    
print("📖 После добавления:")
print(note.read_text(encoding="utf-8"))

## 🔹 4. Обход директорий и поиск
`iterdir()` → плоский список  
`rglob()` → рекурсивный поиск по маске

In [ ]:
root = Path("demo_workspace/raw_data")

print("📂 Содержимое папки (iterdir):")
for item in root.iterdir():
    size = item.stat().st_size
    print(f"  {'📁' if item.is_dir() else '📄'} {item.name} ({size} B)")

print("\n🔍 Рекурсивный поиск CSV и JSON:")
for f in Path("demo_workspace").rglob("*.*"):
    if f.suffix in (".csv", ".json") and f.is_file():
        print(f"  Найдено: {f.relative_to(WORKSPACE)}")

## 🔹 5. `os` vs `pathlib`: когда что использовать
`pathlib` — стандарт для путей. `os` — для системных вызовов.

In [ ]:
import os

# ✅ pathlib (современный)
p = Path("demo_workspace/raw_data/log_01.csv")
print(f"pathlib: {p.exists()}")

# ⚠️ os.path (legacy, но жив)
print(f"os.path: {os.path.isfile(p)}")

# 🛠 Когда os всё ещё нужен:
print(f"\nПеременные окружения: PATH={os.getenv('PATH')}")
print(f"Текущий PID процесса: {os.getpid()}")
# os.chmod(), os.system(), os.walk() — специфичные случаи

## 🔹 6. Безопасность и обработка ошибок
Никогда не доверяйте пользовательскому вводу без валидации.

In [ ]:
# 1. Безопасное удаление
fake = Path("demo_workspace/will_delete.txt")
fake.touch()
fake.unlink()  # удаляет файл
# fake.unlink(missing_ok=True)  # Python 3.8+, не кидает ошибку если файла нет

# 2. Обработка ошибок
try:
    p = Path("demo_workspace/non_existent.txt")
    content = p.read_text()
except FileNotFoundError:
    print("❌ Файл не найден (ожидаемо)")
except PermissionError:
    print("🔒 Нет прав на чтение")

# 3. Защита от Path Traversal
user_input = "../../etc/passwd"  # ⚠️ у вас пытаются украсть пароль 
safe_base = Path("demo_workspace/raw_data")
candidate = (safe_base / user_input).resolve()

print(f"\n🔒 Проверка выхода за пределы:")
print(f"Кандидат: {candidate}")
print(f"База: {safe_base.resolve()}")
print(f"Валиден?", candidate.is_relative_to(safe_base.resolve()))

## 🔹 7. Копирование и перемещение
`pathlib` до 3.14 не умеет копировать. Используем `shutil`.

In [ ]:
src = Path("demo_workspace/raw_data/log_01.csv")
dst_dir = Path("demo_workspace/archive")

# Копирование с сохранением метаданных (время изменения, права)
dst_file = dst_dir / src.name
shutil.copy2(src, dst_file)
print(f"✅ Скопировано: {dst_file.exists()}")
print(f"   Метаданные совпадают? {src.stat().st_mtime == dst_file.stat().st_mtime}")
print(src.stat().st_mtime)

# Перемещение (атомарно на одной ФС)
new_name = dst_dir / "final_report.csv"

if new_name.exists():
    new_name.unlink()
dst_file.rename(new_name)
print(f"✅ Переименовано/перемещено: {new_name.exists()}")
print(f"   Старый файл удалён? {dst_file.exists()}")

In [ ]:
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
    print("🗑️ Рабочая папка demo_workspace/ удалена")
else:
    print("ℹ️ Папка уже отсутствует")